Paquetes necesarios

In [1]:
import cv2  
import csv
from ultralytics import YOLO
from paddleocr import PaddleOCR
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
import re
# Vainas
import time
import platform

import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

c:\Users\skyri\anaconda3\envs\VC_P4\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

# 1. Carga de los modelos 
DISPOSITIVO = "cuda"

detector = YOLO("matriculas_best.pt")

paddle = PaddleOCR(
    lang='es',
    enable_mkldnn=False,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
)

VLM_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
processor = AutoProcessor.from_pretrained(VLM_ID)
vlm = AutoModelForImageTextToText.from_pretrained(VLM_ID, dtype=torch.float32).to(DISPOSITIVO)

PROMPT = "Read the license plate in this image. Answer only with the letters and numbers of the plate."


# 2. Funciones de lectura de matrícula
def limpiar(texto):
    return re.sub(r"[^A-Z0-9]", "", texto.upper())


def leer_paddle(img):
    resultado = paddle.predict(img)[0]
    return limpiar("".join(resultado["rec_texts"]))


def leer_matricula_vlm(img):
    imagen = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    mensajes = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
    prompt = processor.apply_chat_template(mensajes, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[imagen], return_tensors="pt").to(DISPOSITIVO, dtype=torch.float32)
    salida = vlm.generate(**inputs, max_new_tokens=15, do_sample=False)
    nuevos = salida[:, inputs["input_ids"].shape[1]:]
    texto = processor.batch_decode(nuevos, skip_special_tokens=True)[0].strip()
    return limpiar(texto)

def recortar_matricula(img):
    deteccion = detector(img, verbose=False)[0]
    if len(deteccion.boxes[0]) == 0:
        return None
    caja = deteccion.boxes[int(deteccion.boxes.conf.argmax())]
    x1, y1, x2, y2 = (int(v) for v in caja.xyxy[0])
    recorte = img[y1:y2, x1:x2]

    factor = max(1.0, 80 / recorte.shape[0])
    recorte = cv2.resize(recorte, None, fx=factor, fy=factor, interpolation=cv2.INTER_CUBIC)
    return cv2.copyMakeBorder(recorte, 10, 10, 10, 10, cv2.BORDER_REPLICATE)


# 3. Carga de las imagenes de test y las matriculas reales.
CARPETA = "fotos"

datos = []
for nombre in os.listdir(CARPETA):
    if nombre.lower().endswith((".jpg", ".jpeg", ".png")):
        real = limpiar(os.path.splitext(nombre)[0].split("_")[0])
        datos.append((nombre, real))

print("Imágenes de prueba:", len(datos))


# 4. Procesamiento de las imágenes
ocrs = {"PaddleOCR": leer_paddle, "SmolVLM": leer_matricula_vlm}

primera = recortar_matricula(cv2.imread(os.path.join(CARPETA, datos[0][0])))
for leer in ocrs.values():
    leer(primera)

filas = []   
for nombre_imagen, real in datos:
    img = cv2.imread(os.path.join(CARPETA, nombre_imagen))
    recorte = recortar_matricula(img)
    if recorte is None:
        print("Sin matrícula detectada:", nombre_imagen)
        continue

    for nombre_ocr, leer in ocrs.items():
        inicio = time.perf_counter()
        leida = leer(recorte)                             
        tiempo_ms = (time.perf_counter() - inicio) * 1000
        filas.append([nombre_imagen, nombre_ocr, real, leida, int(leida == real), round(tiempo_ms, 1)])

with open("resultados_comparativa.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f, delimiter=";")
    writer.writerow(["imagen", "ocr", "real", "leida", "acierto", "tiempo_ms"])
    writer.writerows(filas)


# 5.Métricas
aciertos, tiempos = {}, {}
for nombre_ocr in ocrs:
    mis_filas = [fila for fila in filas if fila[1] == nombre_ocr]
    aciertos[nombre_ocr] = 100 * sum(fila[4] for fila in mis_filas) / len(mis_filas)
    tiempos[nombre_ocr] = sum(fila[5] for fila in mis_filas) / len(mis_filas)
    print(f"{nombre_ocr}: {len(mis_filas)} imágenes | acierto {aciertos[nombre_ocr]:.1f} % "
          f"| tiempo medio {tiempos[nombre_ocr]:.0f} ms")

# 6. Información del sistema
print("Sistema:", platform.platform())
print("CPU:", platform.processor())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ninguna")
print("Dispositivo usado en la prueba:", DISPOSITIVO)

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\skyri\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\skyri\.paddlex\official_models\PP-OCRv6_medium_rec`.
Loading weights: 100%|██████████| 471/471 [00:00<00:00, 2368.38it/s]


Imágenes de prueba: 45
PaddleOCR: 45 imágenes | acierto 42.2 % | tiempo medio 777 ms
SmolVLM: 45 imágenes | acierto 86.7 % | tiempo medio 534 ms
Sistema: Windows-10-10.0.26200-SP0
CPU: AMD64 Family 25 Model 117 Stepping 2, AuthenticAMD
GPU: NVIDIA GeForce RTX 5060
Dispositivo usado en la prueba: cuda
